# Sprint 1 - Web scraping and property locations

This notebook establishes the Sprint 1 pipeline for MAST30034 Project 2. The current project decision is to plan a three-year rental-price forecast. This notebook does not claim to forecast future prices yet: it validates the listing schema, parser, and location map first.

The local HTML file is a synthetic fixture used only for testing. For the Sprint 1 map, we use the public Kaggle rental snapshot filtered to Victoria while the authorised live source is being confirmed.

In [1]:
from pathlib import Path
import sys
import pandas as pd

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(path for path in candidates if (path / 'src' / 'scraper.py').exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.scraper import extract_listing_records, records_to_frame
from src.integration import load_victoria_data_layers
from src.visualisation import create_enriched_property_map

FORECAST_HORIZON_YEARS = 3
FIXTURE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'example_listing_page.html'
KAGGLE_CSV_PATH = PROJECT_ROOT / 'data' / 'external' / 'australian-rental-market-data-2026' / 'australian_rental_market_2026.csv'
print(PROJECT_ROOT)
print(f'Forecast horizon: {FORECAST_HORIZON_YEARS} years')

C:\Users\86180\Desktop\Applied Data Science\project-2-real-estate
Forecast horizon: 3 years


## 1. Define the property-level schema

The target for the later rental model is `weekly_rent_aud`. The remaining fields are candidate internal features or identifiers for joining external data.

In [2]:
schema = {
    'weekly_rent_aud': 'weekly asking rent in AUD',
    'bedrooms': 'number of bedrooms',
    'bathrooms': 'number of bathrooms',
    'parking_spaces': 'number of parking spaces',
    'property_type': 'house, apartment, townhouse, etc.',
    'suburb': 'suburb used for SA2 and business aggregation',
    'latitude': 'property latitude where provided',
    'longitude': 'property longitude where provided',
}
pd.Series(schema, name='definition').to_frame()

,definition
weekly_rent_aud,weekly asking rent in AUD
bedrooms,number of bedrooms
bathrooms,number of bathrooms
parking_spaces,number of parking spaces
property_type,"house, apartment, townhouse, etc."
suburb,suburb used for SA2 and business aggregation
latitude,property latitude where provided
longitude,property longitude where provided


## 2. Test the parser on the local fixture

In [3]:
html = FIXTURE_PATH.read_text(encoding='utf-8')
records = extract_listing_records(html, source_page_url=FIXTURE_PATH.as_uri())
fixture_listings = records_to_frame(records)
fixture_listings

,listing_id,source,source_page_url,listing_url,collected_at_utc,title,address,suburb,state,postcode,property_type,weekly_rent_aud,bedrooms,bathrooms,parking_spaces,latitude,longitude
0,3caaba066aa3ea91,local_fixture,file:///C:/Users/86180/Desktop/Applied%20Data%...,https://example.invalid/listing/1,2026-09-03T07:16:44.259748+00:00,"1 Example Street, Richmond VIC 3121","1 Example Street, Richmond, VIC, 3121",Richmond,VIC,3121,Apartment,620.0,2,1,1,-37.8183,144.9988
1,a119adb93b01b7f2,local_fixture,file:///C:/Users/86180/Desktop/Applied%20Data%...,https://example.invalid/listing/2,2026-09-03T07:16:44.259748+00:00,"2 Sample Road, Geelong VIC 3220","2 Sample Road, Geelong, VIC, 3220",Geelong,VIC,3220,House,540.0,3,2,2,-38.1499,144.3617
2,34075e3c58554a7d,local_fixture,file:///C:/Users/86180/Desktop/Applied%20Data%...,https://example.invalid/listing/3,2026-09-03T07:16:44.259748+00:00,"3 Test Avenue, Ballarat VIC 3350","3 Test Avenue, Ballarat VIC 3350",Ballarat,VIC,NaN,NaN,480.0,3,1,2,NaN,NaN


In [4]:
assert len(fixture_listings) == 3
assert fixture_listings['weekly_rent_aud'].notna().all()
assert fixture_listings['listing_id'].is_unique
assert fixture_listings['state'].dropna().eq('VIC').all()
print('Parser smoke test passed:', len(fixture_listings), 'synthetic records')

Parser smoke test passed: 3 synthetic records


## 3. Visualise listing locations

The map uses exactly the 1,118 Victoria rows from the Kaggle snapshot. Each geocoded listing is assigned to an official ABS ASGS 2021 SA2 polygon. Homes Victoria suburb history and quarterly benchmarks, plus the Anglicare affordability summary, are displayed only as clearly labelled context; they do not create extra listing observations.

In [5]:
layers = load_victoria_data_layers(PROJECT_ROOT)
listings = layers['kaggle_listings_vic_2026']
history = layers['homes_victoria_suburb_history']
affordability = layers['anglicare_affordability']
benchmarks = layers['homes_victoria_quarterly_benchmarks']
sa2_boundaries = layers['abs_sa2_boundaries']
assert listings['state'].eq('VIC').all()
assert listings['listing_id'].is_unique
assert listings['sa2_code'].notna().all()
print('Loaded Kaggle Victoria listings:', len(listings))
print('Listings assigned to SA2s:', listings['sa2_code'].nunique())

map_path = PROJECT_ROOT / 'data' / 'processed' / 'sprint1_sa2_victoria_property_map.html'
property_map = create_enriched_property_map(
    listings,
    map_path,
    history=history,
    affordability=affordability,
    benchmarks=benchmarks,
    sa2_boundaries=sa2_boundaries,
)
print(f'Saved SA2 map to {map_path}')
property_map

Loaded Kaggle Victoria listings: 1118
Listings assigned to SA2s: 344


Saved SA2 map to C:\Users\86180\Desktop\Applied Data Science\project-2-real-estate\data\processed\sprint1_sa2_victoria_property_map.html


## 4. Live collection: keep opt-in

Fill `AUTHORISED_SOURCE_URLS` only after the group has confirmed that automated access is allowed. `crawl_pages` checks `robots.txt`, uses a descriptive user agent, spaces requests, and refuses disallowed pages. Never bypass access controls or rate limits.

In [6]:
# Keep empty until an authorised source has been confirmed.
AUTHORISED_SOURCE_URLS = []

if AUTHORISED_SOURCE_URLS:
    from src.scraper import crawl_pages, save_records
    live_records = crawl_pages(AUTHORISED_SOURCE_URLS, min_delay_seconds=2.0)
    live_listings = save_records(live_records, PROJECT_ROOT / 'data' / 'processed' / 'sprint1_listings.csv')
    live_listings.head()
else:
    print('Live collection is disabled: confirm an authorised source before adding URLs.')

Live collection is disabled: confirm an authorised source before adding URLs.
